# Analiza portfela XTB

Wczytaj eksporty CSV do `data/`, uruchom wszystkie komórki. Wykresy pokazują:
- wartość portfela z widocznymi skokami wpłat,
- porównanie z S&P 500 (gdyby każda wpłata szła w indeks),
- koszt przewalutowań (0,5% + spread vs NBP),
- aktualną stopę zwrotu.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from xtb_analyzer.benchmark import build_sp500_deposit_benchmark
from xtb_analyzer.charts import (
    plot_benchmark_comparison,
    plot_conversion_losses,
    plot_nav_with_deposits,
    plot_return_dashboard,
)
from xtb_analyzer.flows import analyze_conversion_losses, attach_deposits_to_nav, conversion_summary
from xtb_analyzer.metrics import build_report, compute_metrics, drawdown_series, realized_pnl_summary, rolling_volatility
from xtb_analyzer.parser import load_accounts
from xtb_analyzer.portfolio import build_nav_series
from xtb_analyzer.sectors import enrich_positions_with_sectors, sector_allocation

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../output")
REPORT_CURRENCY = "pln"  # pln | usd | eur

CSV_FILES = sorted(DATA_DIR.glob("XTB_*_account_cs.csv"))
if not CSV_FILES:
    raise FileNotFoundError("Brak plików CSV w data/. Wklej eksporty XTB.")

accounts = load_accounts(CSV_FILES)
nav, holdings_ts, positions = build_nav_series(accounts)
nav = attach_deposits_to_nav(nav, accounts)
positions = enrich_positions_with_sectors(positions)
conversion_losses = analyze_conversion_losses(accounts)
conversion_stats = conversion_summary(conversion_losses)
sp500_benchmark = build_sp500_deposit_benchmark(accounts, nav.index, currency=REPORT_CURRENCY)

print(f"Konta: {[a.account_id for a in accounts]}")
print(f"Okres: {nav.index[0].date()} → {nav.index[-1].date()}")

Konta: ['55287715', '53287226']
Okres: 2025-11-04 → 2026-06-26


## Aktualna stopa zwrotu

In [2]:
plot_return_dashboard(nav, REPORT_CURRENCY).show()

## Wartość portfela w czasie (ze skokami wpłat)

In [3]:
plot_nav_with_deposits(nav, REPORT_CURRENCY).show()

## Portfel vs S&P 500 (te same wpłaty w te same dni)

In [4]:
plot_benchmark_comparison(nav, sp500_benchmark, REPORT_CURRENCY).show()

## Koszt przewalutowań (prowizja 0,5% + spread vs NBP)

In [5]:
display(conversion_losses)
plot_conversion_losses(conversion_losses, conversion_stats).show()

,timestamp,account_id,to_currency,pln_spent,foreign_received,xtb_rate,nbp_rate,fair_foreign_at_mid,fair_foreign_after_fee,commission_loss_pln,spread_loss_pln,total_loss_pln,cumulative_loss_pln
0,2025-11-04 21:48:00,53287226,USD,4499.986943,1206.28,0.268063,3.7019,1215.588466,1209.510524,22.499935,34.459011,56.958946,56.958946
1,2025-11-06 19:50:00,53287226,USD,1999.977805,540.65,0.270328,3.6971,540.958536,538.253744,9.999889,1.140690,11.140579,68.099525
2,2025-11-21 15:16:00,53287226,USD,999.985175,269.81,0.269814,3.6868,271.233909,269.877739,4.999926,5.249667,10.249593,78.349118
3,2025-12-11 17:40:00,53287226,USD,699.982650,193.66,0.276664,3.6132,193.729284,192.760638,3.499913,0.250338,3.750252,82.099369
4,2026-01-22 21:35:00,53287226,USD,599.964797,167.02,0.278383,3.6028,166.527367,165.694730,2.999824,0.000000,2.999824,85.099193
5,2026-04-22 19:27:00,53287226,USD,274.970306,75.47,0.274466,3.6080,76.211282,75.830226,1.374852,2.674546,4.049398,89.148591
6,2026-05-27 14:34:41,55287715,EUR,749.972329,176.17,0.234902,4.2393,176.909473,176.024926,3.749862,3.134848,6.884710,96.033300
7,2026-06-17 21:02:00,53287226,USD,676.899747,183.01,0.270365,3.6542,185.238834,184.312640,3.384499,8.144605,11.529103,107.562404
8,2026-06-22 15:46:00,53287226,USD,337.950805,89.95,0.266163,3.7282,90.647177,90.193941,1.689754,2.599215,4.288969,111.851372


## Pozycje, sektory i metryki ryzyka

In [6]:
display(positions.sort_values("market_value_pln", ascending=False))
display(sector_allocation(positions))
display(pd.DataFrame([compute_metrics(nav, REPORT_CURRENCY).__dict__]))
display(realized_pnl_summary(accounts))

,account_id,symbol,currency,shares,avg_cost,market_price,cost_basis,market_value,unrealized_pnl,market_value_pln,market_value_usd,market_value_eur,weight,sector,industry
4,53287226,NVDA.US,USD,5.6500,210.704071,192.529999,1190.478000,1087.794493,-102.683507,4089.998515,1087.794493,955.083588,0.294331,Technology,Semiconductors
0,53287226,UBER.US,USD,13.0000,84.765286,76.199997,1101.948712,990.599960,-111.348752,3724.556791,990.599960,869.746787,0.268033,Technology,Software - Application
2,53287226,NOW.US,USD,5.0000,103.920124,98.339996,519.600622,491.699982,-27.900640,1848.742761,491.699982,431.712595,0.133042,Technology,Software - Application
1,53287226,GOOGL.US,USD,1.0000,366.640445,337.390015,366.640445,337.390015,-29.250430,1268.552716,337.390015,296.228440,0.091290,Communication Services,Internet Content & Information
8,53287226,CRDO.US,USD,1.0000,221.920000,238.000000,221.920000,238.000000,16.080000,894.856200,238.000000,208.964005,0.064397,Technology,Semiconductors
3,55287715,IFX.DE,EUR,2.2546,78.058871,78.300003,175.991530,176.535187,0.543657,756.788693,201.065128,176.535187,0.054461,Technology,Semiconductors
5,53287226,NU.US,USD,12.6000,12.923492,13.170000,162.836000,165.942001,3.106001,623.925329,165.942001,145.697080,0.044900,Financial Services,Banks - Regional
7,53287226,ACMR.US,USD,1.3006,83.002357,104.500000,107.952866,135.912700,27.959834,511.018161,135.912700,119.331354,0.036775,Technology,Semiconductor Equipment & Materials
6,53287226,POET.US,USD,5.0000,12.790000,9.440000,63.950000,47.199998,-16.750002,177.467272,47.199998,41.441599,0.012771,Technology,Semiconductors


,sector,market_value_pln,unrealized_pnl,symbols,weight
2,Technology,12003.428392,-214.099410,7,0.86381
0,Communication Services,1268.552716,-29.250430,1,0.09129
1,Financial Services,623.925329,3.106001,1,0.04490


,total_return,cagr,volatility,sharpe,sortino,max_drawdown,max_drawdown_start,max_drawdown_end,ath,pct_from_ath,beta,alpha,benchmark_total_return
0,2.119521,4.905008,0.682421,2.836446,7.684315,-0.190424,2026-01-23,2026-02-23,14172.20657,-0.019075,1.551775,1.727322,0.086017


,symbol,trades,total_realized_pnl,avg_realized_pnl,win_rate
0,AMD.US,10,1012.08,101.2080,1.00
1,AMZN.US,4,-2.99,-0.7475,0.25


## Eksport

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
build_report(accounts, nav, positions, output_dir=OUTPUT_DIR)
nav.to_csv(OUTPUT_DIR / "nav_timeseries.csv")
conversion_losses.to_csv(OUTPUT_DIR / "conversion_losses.csv", index=False)
sp500_benchmark.to_csv(OUTPUT_DIR / "sp500_benchmark.csv")
print(f"Zapisano do {OUTPUT_DIR.resolve()}")

Zapisano do C:\Users\oliwi\OneDrive\Desktop\xtb_portfolio_analyzer\output
